# 10주차 워크북 1 — Cloud Vision API 요청·응답 이해하기 (Colab 편)

ESP32 예제 3·4가 Vision에 보내는 **요청 JSON**과 받는 **응답 JSON**을 Colab에서 먼저 눈으로 확인한다.
ESP32 시리얼 모니터에 찍히는 숫자(이미지 크기, Base64 길이, 라벨·점수)와 비교하면서 진행한다.

- 강의자료: [10주차 학습자료(Notion)](https://app.notion.com/p/40db2e6bf80780f0b253edeeb0253a31)
- 실행: 위에서부터 `Shift+Enter`. **학생용**은 `____` 빈칸을 채워야 다음 셀이 동작한다.
- API 키: 강의자가 수업 시간에 알려 준 키를 입력창에 붙여넣는다(노트북에 저장되지 않음). 키가 없으면 Enter → 캐시 모드.

![ESP32 Vision 흐름](https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week10_cloud_vision/diagrams/05_esp32_vision_flow.png)

## 핵심 키워드 미리 보기

| 키워드 | 뜻 |
| --- | --- |
| `gdown.download_folder(id=...)` | 공유된 Google Drive 폴더를 Colab으로 내려받기 |
| `Path.read_bytes()` | 파일을 바이트(0~255 숫자열)로 읽기 |
| `base64.b64encode()` | 바이트 → 글자(A~Z, a~z, 0~9, +, /)로 변환, 길이는 약 4/3배 |
| `LABEL_DETECTION` | Vision 기능 이름: 이미지에 어울리는 라벨 여러 개 반환 |
| `maxResults` | 받을 라벨 최대 개수 |
| `requests.post(url, params, json)` | HTTPS POST 요청 (ESP32의 `http.POST(body)`와 같은 일) |
| `labelAnnotations[i].description / .score` | 라벨 이름 / 신뢰도(0~1) |

In [ ]:
!pip -q install gdown

## 1. 데이터 준비 — Drive에 미리 올려 둔 과일 80장 내려받기

**▶ 실행 결과 미리 보기**

```text
데이터 출처 : Google Drive
이미지 수   : 80
첫 3줄      : [{'path': 'images/apple/apple_01.jpg', 'true_label': 'apple'}, {'path': 'images/apple/apple_02.jpg', 'true_label': 'apple'}, {'path': 'images/apple/apple_03.jpg', 'true_label': 'apple'}]
```

In [ ]:
# 데이터 준비: ① 강의 Google Drive 폴더 → ② GitHub → ③ 직접 업로드 순서로 시도
import csv
import json
import os
import urllib.request
import zipfile
from pathlib import Path

DATA_FOLDER_ID = "1Is7dwskGMYvOxeIRhy0QYMZejj0K8BWE"   # Drive: 10주_강의자료/04_데이터_과일80장
GITHUB_ZIP = (
    "https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/"
    "main/week10_cloud_vision/dataset/week10_fruits.zip"
)
DATA_DIR = Path("week10_fruits")      # 압축을 풀면 생기는 폴더
DRIVE_DIR = Path("week10_drive")      # Drive 폴더를 내려받는 곳
CLASSES = ["apple", "banana", "orange", "lemon"]


def prepare_data():
    """과일 80장 데이터셋을 준비하고, 어디서 받았는지 돌려준다."""
    if (DATA_DIR / "labels.csv").exists():
        return "이미 준비됨"

    zip_path, source = None, None

    # ① Google Drive 폴더 (강의자가 미리 올려 둔 파일, 가장 빠름)
    try:
        import gdown
        gdown.download_folder(id=DATA_FOLDER_ID, output=str(DRIVE_DIR), quiet=True)
        found = list(DRIVE_DIR.rglob("week10_fruits.zip"))
        if found:
            zip_path, source = found[0], "Google Drive"
    except Exception as e:
        print("Drive 다운로드 실패 →", type(e).__name__)

    # ② GitHub (Drive가 막혔을 때)
    if zip_path is None:
        try:
            urllib.request.urlretrieve(GITHUB_ZIP, "week10_fruits.zip")
            zip_path, source = Path("week10_fruits.zip"), "GitHub"
        except Exception as e:
            print("GitHub 다운로드 실패 →", type(e).__name__)

    # ③ 직접 업로드 (왼쪽 파일 창에 week10_fruits.zip 을 끌어다 놓기)
    if zip_path is None and Path("week10_fruits.zip").exists():
        zip_path, source = Path("week10_fruits.zip"), "직접 업로드"

    if zip_path is None:
        raise FileNotFoundError("week10_fruits.zip 을 왼쪽 파일 창에 업로드한 뒤 다시 실행하세요")

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    return source


source = prepare_data()
rows = list(csv.DictReader(open(DATA_DIR / "labels.csv", encoding="utf-8")))
print("데이터 출처 :", source)
print("이미지 수   :", len(rows))
print("첫 3줄      :", rows[:3])

## 2. 이미지 1장을 바이트와 Base64로 보기 (ESP32 예제 2와 같은 일)

ESP32 예제 2가 시리얼에 출력한 `크기`, `첫 3바이트`, `Base64 길이`와 같은 값이 나오는지 비교한다.

**▶ 실행 결과 미리 보기**

```text
파일       : images/apple/apple_02.jpg
크기       : 4925 bytes
첫 3바이트 : FF D8 FF (FF D8 FF 이면 JPEG)
Base64 길이: 6568 글자 = 1.33 배
Base64 앞부분: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAIBAQEB...
이미지 크기: (100, 100)
```

In [ ]:
import base64
from PIL import Image
import matplotlib.pyplot as plt

path = "images/apple/apple_02.jpg"          # ESP32 예제2·3의 첫 번째 이미지
raw = (DATA_DIR / path).read_bytes()
b64 = base64.b64encode(raw).decode("ascii")   # 빈칸: base64 변환

print("파일       :", path)
print("크기       :", len(raw), "bytes")
print("첫 3바이트 :", raw[:3].hex(" ").upper(), "(FF D8 FF 이면 JPEG)")
print("Base64 길이:", len(b64), "글자 =", round(len(b64) / len(raw), 2), "배")
print("Base64 앞부분:", b64[:40] + "...")

img = Image.open(DATA_DIR / path)
print("이미지 크기:", img.size)
plt.figure(figsize=(2, 2))
plt.imshow(img)
plt.axis("off")
plt.show()

## 3. 요청 JSON 만들기 (ESP32 예제 3의 `body`와 같은 구조)

Vision은 `{"requests": [ {이미지, 기능}, ... ]}` 형태를 받는다. 한 번에 여러 장(최대 16장)을 보낼 수 있다.

**▶ 실행 결과 미리 보기**

```text
요청 JSON 길이: 6670 글자
{"requests": [{"image": {"content": "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDA ... features": [{"type": "LABEL_DETECTION", "maxResults": 5}]}]}
```

In [ ]:
request_body = {
    "requests": [{
        "image": {"content": b64},
        "features": [{"type": "LABEL_DETECTION", "maxResults": 5}],
    }]
}
text = json.dumps(request_body)
print("요청 JSON 길이:", len(text), "글자")
print(text[:70] + " ... " + text[-60:])

## 4. API 키 준비와 모드 확인

- `live` : 키를 입력해 Vision을 직접 호출 (수업 권장)
- `cache` : 키 없이 Enter → 강의자가 미리 저장한 결과(`vision_cache.json`) 사용
- Colab 왼쪽 🔑(보안 비밀)에 `VISION_API_KEY`를 저장해 두면 입력창 없이 사용된다

**▶ 실행 결과 미리 보기**

```text
cache 모드로 실행했을 때:
Vision 모드: cache
  live  = API를 직접 호출 (키 있음)
  cache = 강의자가 미리 저장한 Vision 결과 사용 (키 없음)
  none  = Vision 단계 건너뜀 (분할·kNN 실습은 그대로 가능)
캐시 정보: {'created': '2026-11-..', 'feature': 'LABEL_DETECTION', 'maxResults': 10, 'images': 80}
```

In [ ]:
# Cloud Vision REST 호출 함수 + API 키 / 캐시 준비
import base64
import requests

VISION_URL = "https://vision.googleapis.com/v1/images:annotate"
GITHUB_CACHE = (
    "https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/"
    "main/week10_cloud_vision/dataset/vision_cache.json"
)


def get_api_key():
    """① Colab 보안 비밀(VISION_API_KEY) → ② 입력창(getpass) 순서로 키를 받는다."""
    if os.environ.get("VISION_API_KEY"):
        return os.environ["VISION_API_KEY"]
    try:
        from google.colab import userdata
        key = userdata.get("VISION_API_KEY")
        if key:
            return key
    except Exception:
        pass
    try:
        from getpass import getpass
        return getpass("Cloud Vision API 키 (없으면 그냥 Enter): ").strip()
    except Exception:
        return ""


def vision_labels(paths, api_key, max_results=10, batch=16):
    """이미지 경로 목록 → {경로: [{description, score}, ...]} (요청 1번에 최대 16장)"""
    result = {}
    for i in range(0, len(paths), batch):
        chunk = paths[i:i + batch]
        requests_body = []
        for p in chunk:
            content = base64.b64encode((DATA_DIR / p).read_bytes()).decode("ascii")
            requests_body.append({
                "image": {"content": content},
                "features": [{"type": "LABEL_DETECTION", "maxResults": max_results}],
            })
        r = requests.post(VISION_URL, params={"key": api_key},
                          json={"requests": requests_body}, timeout=60)
        data = r.json()
        if r.status_code != 200:
            raise RuntimeError(data.get("error", {}).get("message", r.text[:200]))
        for p, resp in zip(chunk, data["responses"]):
            if "error" in resp:
                raise RuntimeError(f"{p}: {resp['error'].get('message')}")
            result[p] = [
                {"description": a["description"], "score": round(a["score"], 4)}
                for a in resp.get("labelAnnotations", [])
            ]
    return result


def load_cache():
    """강의자가 만든 vision_cache.json (Drive 폴더 → GitHub 순서)"""
    found = list(DRIVE_DIR.rglob("vision_cache.json")) + [
        p for p in [Path("vision_cache.json")] if p.exists()]
    if not found:
        try:
            urllib.request.urlretrieve(GITHUB_CACHE, "vision_cache.json")
            found = [Path("vision_cache.json")]
        except Exception:
            return None
    return json.loads(found[0].read_text(encoding="utf-8"))


API_KEY = get_api_key()
CACHE = load_cache()
if API_KEY:
    MODE = "live"
elif CACHE:
    MODE = "cache"
else:
    MODE = "none"

print("Vision 모드:", MODE)
print("  live  = API를 직접 호출 (키 있음)")
print("  cache = 강의자가 미리 저장한 Vision 결과 사용 (키 없음)")
print("  none  = Vision 단계 건너뜀 (분할·kNN 실습은 그대로 가능)")
if CACHE:
    print("캐시 정보:", CACHE["meta"])

def get_labels(paths):
    """모드에 맞게 라벨을 가져온다."""
    if MODE == "live":
        return vision_labels(paths, API_KEY)
    if MODE == "cache":
        return {p: CACHE["labels"][p] for p in paths}
    return {}

## 5. 응답 JSON 살펴보기 — 이미지 1장

`live` 모드면 실제 응답 원문을, `cache` 모드면 저장된 라벨을 같은 모양으로 보여 준다.

**▶ 실행 결과 미리 보기** — 형식 예시: 실제 Vision 라벨·점수·정확도는 실행할 때 달라집니다

```text
(cache 모드: 저장된 결과를 응답 모양으로 표시)
{
 "responses": [
  {
   "labelAnnotations": [
    {
     "description": "Apple",
     "score": 0.9405
    },
    {
     "description": "Fruit",
     "score": 0.9364
    },
    {
     "description": "Natural foods",
     "score": 0.8673
    },
    {
     "description": "Mcintosh",
     "score": 0.7829
    },
    {
     "description": "Plant",
     "score": 0.7725
    }
   ]
  }
 ]
}

순위  description               score
  1   Apple                   0.941
  2   Fruit                   0.936
  3   Natural foods           0.867
  4   Mcintosh                0.783
  5   Plant                   0.772
```

In [ ]:
if MODE == "live":
    r = requests.post(VISION_URL, params={"key": API_KEY}, json=request_body, timeout=60)
    response = r.json()
    print("HTTP 상태 코드:", r.status_code)
elif MODE == "cache":
    response = {"responses": [{"labelAnnotations": CACHE["labels"][path][:5]}]}
    print("(cache 모드: 저장된 결과를 응답 모양으로 표시)")
else:
    response = {"responses": [{"labelAnnotations": []}]}
    print("(Vision 없음: 키 또는 캐시가 필요)")

print(json.dumps(response, ensure_ascii=False, indent=1)[:600])

print("\n순위  description               score")
for i, a in enumerate(response["responses"][0].get("labelAnnotations", []), 1):
    name = a["description"]
    score = a["score"]          # 빈칸: 신뢰도 꺼내기
    print(f"{i:>3}   {name:<24}{score:.3f}")

## 6. 클래스별 1장씩 — ESP32 예제 3과 같은 4장

라벨 중 **어떤 단어가 그 과일을 가리키는지** 표시해 보자. 이것이 워크북 2의 '매핑 규칙'의 출발점이다.

**▶ 실행 결과 미리 보기** — 형식 예시: 실제 Vision 라벨·점수·정확도는 실행할 때 달라집니다

```text
apple   → Apple(0.94), Fruit(0.94), Natural foods(0.87), Mcintosh(0.78), Plant(0.77)
banana  → Banana(0.97), Cooking plantain(0.84), Fruit(0.80), Saba banana(0.76)
orange  → Citrus(0.94), Orange(0.91), Mandarin orange(0.88), Tangerine(0.79), Fruit(0.78)
lemon   → Lemon(0.90), Citrus(0.86), Sweet lemon(0.78), Fruit(0.76), Yellow(0.76)
```

In [ ]:
four = ["images/apple/apple_02.jpg", "images/banana/banana_01.jpg",
        "images/orange/orange_01.jpg", "images/lemon/lemon_01.jpg"]
labels4 = get_labels(four)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
for ax, p in zip(axes, four):
    ax.imshow(Image.open(DATA_DIR / p))
    ax.set_title(p.split("/")[-1], fontsize=9)
    ax.axis("off")
plt.show()

for p in four:
    top = labels4.get(p, [])[:5]
    print(p.split("/")[1].ljust(7), "→", ", ".join(f"{a['description']}({a['score']:.2f})" for a in top))

## 7. 정리 — score 와 accuracy 는 다르다

![score vs accuracy](https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week10_cloud_vision/diagrams/08_confusion_accuracy.png)

| 질문 | 내 답 |
| --- | --- |
| apple_02.jpg 의 크기(bytes)와 Base64 길이는? ESP32 예제2 출력과 같은가? | |
| Vision이 준 1위 라벨이 과일 이름이 아닌 이미지는? 그 라벨은? | |
| lemon 사진의 라벨 중 orange 와 헷갈릴 수 있는 라벨은? | |
| `score 0.93` 은 "93% 정확하다"는 뜻인가? 이유는? | |

다음: ESP32 예제 3·4 (Notion 워크북 1) → 워크북 2 (데이터 분할과 평가)